# HPO LunarLander

Optuna hyperparameter optimization for `TunedTrainer` on LunarLander.

The happy path on Colab is labeled HP1 to HP3.

Hardware: L4 GPU is the intended runtime.

## Setup

### Runtime setup

Set up the runtime by running exactly one code cell in this section.

#### Colab

In [ ]:
# HP1
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys

from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

study_dir = Path("/content/drive/MyDrive/rl_lab/hpo")
run_dir = Path("/content/rl_lab/hpo/runs")
drive.mount("/content/drive")
study_dir.mkdir(parents=True, exist_ok=True)
run_dir.mkdir(parents=True, exist_ok=True)

HPO_STUDY_DIR = study_dir
HPO_RUN_DIR = run_dir

#### Local

In [ ]:
# Local setup
from pathlib import Path
import sys

sys.path.insert(0, "../../dqn/src")
sys.path.insert(0, "../src")

HPO_STUDY_DIR = Path("../runs")
HPO_RUN_DIR = Path("../runs/episodes")
HPO_STUDY_DIR.mkdir(parents=True, exist_ok=True)
HPO_RUN_DIR.mkdir(parents=True, exist_ok=True)

## Imports

In [ ]:
# HP2
from IPython.display import clear_output, display
import optuna
import torch
from optuna.visualization import plot_optimization_history

from hpo.lunar_lander.objective import create_objective

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Optimization

In [ ]:
# HP3
objective = create_objective(
    num_episodes=500,
    score_window=50,
    seed=42,
    output_dir=HPO_RUN_DIR,
    device=device,
)

study_db_path = HPO_STUDY_DIR / "lunar_lander_dqn.db"
study = optuna.create_study(
    study_name="lunar_lander_dqn",
    direction="maximize",
    storage=f"sqlite:///{study_db_path}",
    load_if_exists=True,
)
num_trials = 20

for trial_index in range(num_trials):
    study.optimize(objective, n_trials=1)

    clear_output(wait=True)
    best_trial = study.best_trial

    print(f"Trial: {trial_index + 1}/{num_trials}")
    print(f"Completed trials: {len(study.trials)}")
    print(f"Best mean return: {best_trial.value:.1f}")
    print(
        "Best episode window:",
        best_trial.user_attrs["best_window_start_episode"],
        "-",
        best_trial.user_attrs["best_window_end_episode"],
    )
    print("Best params:")
    display(best_trial.params)
    display(plot_optimization_history(study))
